# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

## Setup (Local)

In [1]:
import os
from pathlib import Path

import duckdb

ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    import getpass
    print("Tip: pip install python-dotenv")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
# Feature window (past): Mar–Apr 2026. Outcome window (future label): May 2026.
# Explicit path list — hf:// does not support brace globs.
FACT_PREV = (
    "read_parquet(["
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet', "
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    "])"
)
FACT_NEXT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-05/*.parquet')"

print("Connected.")
print("  FACT_PREV = month=2026-03 + month=2026-04 (score inputs)")
print("  FACT_NEXT = month=2026-05 (outcome label)")

Connected.
  FACT_PREV = month=2026-03 + month=2026-04 (score inputs)
  FACT_NEXT = month=2026-05 (outcome label)


## 1. My rule and its reason codes

**Lane 4 — CTR opportunity scoring**

**Slice:** Mar–Apr 2026 features → May 2026 label. One row per page. Floor: **`imp >= 500`** (features, medians, queue, label). Real position required.

### Rule

> Score = `ctr_gap` (past tier-median CTR − past page CTR). Rank eligible pages (`imp_prev >= 500`) by that score; check who underperforms in **May**.

**Ranking contract** (baseline and model — identical):
1. Score descending (`ctr_gap` / model score)
2. Ties → `imp_prev` descending
3. Pool: `imp_prev >= 500` only
4. Same order for Precision@K — no other tie-break

### Signal checks → verdicts (check code below)

| Signal | Check | Verdict |
|---|---|---|
| **1. CTR vs position** | Does past CTR fall as `position_tier` worsens? | **CONFIRMED** — 0.51% `top_3` → 0.03% `deep` |
| **2. Volume** | Within `>= 500`, are lower bands noisier? | **CONFIRMED** — 28% zero-click in `500-999` → 2% in `3000+` |

### Reason codes

| Code | When | In CSV? |
|---|---|---|
| `high_visibility_ctr_gap` | past CTR below tier median | Yes — priority |
| `general_ctr_monitor` | at/above tier median | Yes — low priority |


In [2]:
import pandas as pd

TIER_ORDER = ["top_3", "page_1", "striking", "page_3_5", "deep"]
IMP_BANDS = [500, 1000, 3000, float("inf")]
IMP_LABELS = ["500-999", "1000-2999", "3000+"]

features = con.sql(f"""
    WITH prev_daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_prev,
            SUM(gsc_clicks) AS clk_prev,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_prev
        FROM {FACT_PREV}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 500
    ),
    prev_scored AS (
        SELECT
            *,
            CASE WHEN imp_prev > 0 THEN 100.0 * clk_prev / imp_prev END AS ctr_prev,
            CASE
                WHEN pos_avg_prev <= 3 THEN 'top_3'
                WHEN pos_avg_prev <= 10 THEN 'page_1'
                WHEN pos_avg_prev <= 20 THEN 'striking'
                WHEN pos_avg_prev <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier
        FROM prev_daily
        WHERE pos_avg_prev > 0
    ),
    prev_ranked AS (
        SELECT
            *,
            MEDIAN(ctr_prev) OVER (PARTITION BY position_tier) AS tier_median_ctr
        FROM prev_scored
    ),
    next_daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_next,
            SUM(gsc_clicks) AS clk_next,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_next
        FROM {FACT_NEXT}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 500
    ),
    next_scored AS (
        SELECT
            *,
            CASE WHEN imp_next > 0 THEN 100.0 * clk_next / imp_next END AS ctr_next,
            CASE
                WHEN pos_avg_next <= 3 THEN 'top_3'
                WHEN pos_avg_next <= 10 THEN 'page_1'
                WHEN pos_avg_next <= 20 THEN 'striking'
                WHEN pos_avg_next <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier_next
        FROM next_daily
        WHERE pos_avg_next > 0
    ),
    next_labeled AS (
        SELECT
            *,
            CASE
                WHEN ctr_next < MEDIAN(ctr_next) OVER (PARTITION BY position_tier_next)
                     AND imp_next >= 500
                THEN 1
                ELSE 0
            END AS is_ctr_underperformer
        FROM next_scored
    )
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.imp_prev,
        p.clk_prev,
        p.pos_avg_prev,
        p.ctr_prev,
        p.position_tier,
        p.tier_median_ctr,
        n.imp_next,
        n.clk_next,
        n.ctr_next,
        n.pos_avg_next,
        n.position_tier_next,
        COALESCE(n.is_ctr_underperformer, 0) AS is_ctr_underperformer
    FROM prev_ranked p
    INNER JOIN next_labeled n
        USING (client_hash_id, content_hash_id)
""").df()

print(f"Lane slice n = {len(features):,} (past features intersect May outcome)")
print(f"Feature window: 2026-03 + 2026-04 | Outcome window: 2026-05")

print("\nSIGNAL 1 — CTR vs position (by past position_tier)")
sig1 = (
    features.groupby("position_tier", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        sum_imp=("imp_prev", "sum"),
        sum_clk=("clk_prev", "sum"),
    )
    .assign(weighted_ctr_prev=lambda d: 100.0 * d["sum_clk"] / d["sum_imp"])
    .drop(columns=["sum_imp", "sum_clk"])
    .reindex(TIER_ORDER)
    .reset_index()
)
print(sig1.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSIGNAL 2 — Volume (by past impression band)")
features["imp_band"] = pd.cut(
    features["imp_prev"], bins=IMP_BANDS, labels=IMP_LABELS, right=False
)
sig2 = (
    features.groupby("imp_band", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        share_zero_clicks=("ctr_prev", lambda s: (s == 0).mean()),
    )
    .reset_index()
)
print(sig2.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


Lane slice n = 52,001 (past features intersect May outcome)
Feature window: 2026-03 + 2026-04 | Outcome window: 2026-05

SIGNAL 1 — CTR vs position (by past position_tier)
position_tier     n  weighted_ctr_prev
        top_3  2385             0.5075
       page_1 28917             0.3271
     striking 10164             0.3269
     page_3_5  9990             0.1373
         deep   545             0.0292

SIGNAL 2 — Volume (by past impression band)
 imp_band     n  share_zero_clicks
  500-999  3908             0.2753
1000-2999 14306             0.1262
    3000+ 33787             0.0228


## 2. Build the ranked queue

*Writes `work/outputs/baseline_action_score.csv`.*

In [4]:
import numpy as np

OUT = ROOT / "work" / "outputs" / "baseline_action_score.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

IMP_FLOOR = 500
queue = features.copy()
queue["ctr_gap"] = queue["tier_median_ctr"] - queue["ctr_prev"]
queue["baseline_score"] = queue["ctr_gap"]
queue["below_tier_median"] = (queue["ctr_prev"] < queue["tier_median_ctr"]).astype(int)


def reason_code(row) -> str:
    if row["below_tier_median"]:
        return "high_visibility_ctr_gap"
    return "general_ctr_monitor"


def action_label(code: str) -> str:
    if code == "high_visibility_ctr_gap":
        return "refresh_and_review_ctr"
    return "monitor"


# Rank eligible pool only (past-window stake floor)
eligible = queue[queue["imp_prev"] >= IMP_FLOOR].copy()
eligible = eligible.sort_values(["baseline_score", "imp_prev"], ascending=[False, False])
eligible["baseline_rank"] = np.arange(1, len(eligible) + 1)
eligible["reason_code"] = eligible.apply(reason_code, axis=1)
eligible["action"] = eligible["reason_code"].map(action_label)

export_cols = [
    "baseline_rank", "content_hash_id", "client_hash_id",
    "imp_prev", "ctr_prev", "pos_avg_prev", "position_tier",
    "imp_next", "ctr_next",
    "baseline_score", "reason_code", "action",
    "is_ctr_underperformer",
]
eligible.to_csv(OUT, index=False, columns=export_cols)
print(f"Wrote {len(eligible):,} eligible rows → {OUT}")

queue = eligible  # downstream cells use the ranked eligible queue


Wrote 52,001 eligible rows → c:\Users\rimla\Desktop\work_folder\flyrank-internship\work\outputs\baseline_action_score.csv


### Baseline metrics

**Metric** = Precision@K — of the top K pages that order ranks, how many underperform in May?

Compare Precision@K to the **base rate** (random pick rate in May). If Precision@K is higher, the rule beats chance.


In [5]:
import json
import numpy as np

LABEL = "is_ctr_underperformer"

def precision_at_k(scores, labels, k, tie_break=None):
    """Rank by score desc, then tie_break desc (same contract for rule and model)."""
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels)
    if tie_break is None:
        order = np.argsort(-scores, kind="mergesort")
    else:
        tb = np.asarray(tie_break, dtype=float)
        order = np.lexsort((-tb, -scores))
    k = min(k, len(order))
    return float(labels[order[:k]].mean())

labels = queue[LABEL].to_numpy()
scores = queue["baseline_score"].to_numpy()
imp = queue["imp_prev"].to_numpy()
base_rate = float(labels.mean())

metrics = {
    "method": "baseline_rule",
    "label": LABEL,
    "slice": "features=2026-03+2026-04; label=2026-05",
    "feature_window": "month=2026-03 + month=2026-04",
    "outcome_window": "month=2026-05",
    "population": f"imp_prev>={IMP_FLOOR}",
    "n": int(len(queue)),
    "base_rate": base_rate,
    "tie_break": "imp_prev",
    "precision_at_10": precision_at_k(scores, labels, 10, tie_break=imp),
    "precision_at_20": precision_at_k(scores, labels, 20, tie_break=imp),
    "precision_at_50": precision_at_k(scores, labels, 50, tie_break=imp),
}

METRICS_OUT = ROOT / "work" / "outputs" / "baseline_metrics.json"
METRICS_OUT.write_text(json.dumps(metrics, indent=2))

print(f"Eligible n = {len(queue):,}")
print(f"Base rate (share {LABEL}=1 in May): {base_rate:.3f}")
for k in (10, 20, 50):
    print(f"Precision@{k}: {metrics[f'precision_at_{k}']:.3f}")
print(f"Saved → {METRICS_OUT}")


Eligible n = 52,001
Base rate (share is_ctr_underperformer=1 in May): 0.515
Precision@10: 0.800
Precision@20: 0.850
Precision@50: 0.780
Saved → c:\Users\rimla\Desktop\work_folder\flyrank-internship\work\outputs\baseline_metrics.json


## 3. Top-10 review

The code cell below prints the top 10. Short read: all are `top_3` + near-zero past CTR + `high_visibility_ctr_gap`. High past impressions on most rows → real review stakes, but zero-click `top_3` may be SERP layout, not a title fix. P@10 = 0.80 (8/10 May underperformers).


In [6]:
top10_cols = [
    "baseline_rank", "content_hash_id", "imp_prev", "ctr_prev",
    "position_tier", "imp_next", "ctr_next",
    "reason_code", "action", "is_ctr_underperformer",
]
print(queue.head(10)[top10_cols].to_string(index=False))


 baseline_rank          content_hash_id  imp_prev  ctr_prev position_tier  imp_next  ctr_next             reason_code                 action  is_ctr_underperformer
             1 content_cc24743a90439e14    8641.0  0.000000         top_3     804.0  0.000000 high_visibility_ctr_gap refresh_and_review_ctr                      1
             2 content_bbcfba786761630c    1685.0  0.000000         top_3     548.0  0.000000 high_visibility_ctr_gap refresh_and_review_ctr                      1
             3 content_2012ece8f2deb671     949.0  0.000000         top_3    1023.0  0.684262 high_visibility_ctr_gap refresh_and_review_ctr                      0
             4 content_1d7764b642f7bb9f   40491.0  0.009879         top_3    2754.0  0.036311 high_visibility_ctr_gap refresh_and_review_ctr                      1
             5 content_307525daee61f1ba    5526.0  0.018096         top_3     635.0  0.157480 high_visibility_ctr_gap refresh_and_review_ctr                      1
             6 c

## 4. Weak picks + leakage check

**Weak picks:**
1. **Rank 3** — max past gap, but May CTR ~0.68% → **label 0** (false priority).
2. **Rank 9** — same story; May CTR ~0.52% → **label 0**.

**Leakage check:**
- No product flags / trends in the score ✓
- Score = Mar–Apr only; May = label only ✓
- No June / `_sample` ✓
- `ctr_gap` is baseline-only (not a model feature) ✓
- Floor `imp >= 500` everywhere ✓


In [7]:
FORBIDDEN = {"trend_direction", "trend_pct", "health_score", "needs_ctr_fix", "priority_score"}
print("Forbidden cols present:", sorted(FORBIDDEN & set(queue.columns)) or "none (good)")

Forbidden cols present: none (good)
